In [5]:
import random
import time


# =========================================================
# TILEWORLD
# =========================================================

GRID_SIZE = 15


class Hole:

    def __init__(self, hole_id, position, lifetime):
        self.id = hole_id
        self.position = position
        self.lifetime = lifetime


class TileWorld:

    def __init__(
        self,
        size=15,
        appearance_probability=0.08,
        min_lifetime=5,
        max_lifetime=20,
        seed=None
    ):

        self.size = size
        self.appearance_probability = appearance_probability
        self.min_lifetime = min_lifetime
        self.max_lifetime = max_lifetime

        self.holes = {}

        self.holes_appeared = 0
        self.holes_filled = 0

        self.next_hole_id = 1

        if seed is not None:
            random.seed(seed)


    def tick(self):

        # ---------------------------------------------
        # Reduce lifetime of existing holes
        # ---------------------------------------------

        expired = []

        for hole_id, hole in self.holes.items():

            hole.lifetime -= 1

            if hole.lifetime <= 0:
                expired.append(hole_id)

        for hole_id in expired:
            del self.holes[hole_id]


        # ---------------------------------------------
        # Randomly create a new hole
        # ---------------------------------------------

        if random.random() < self.appearance_probability:

            x = random.randint(0, self.size - 1)
            y = random.randint(0, self.size - 1)

            position = (x, y)

            # Do not create two holes at same position
            occupied = [
                hole.position
                for hole in self.holes.values()
            ]

            if position not in occupied:

                lifetime = random.randint(
                    self.min_lifetime,
                    self.max_lifetime
                )

                hole = Hole(
                    self.next_hole_id,
                    position,
                    lifetime
                )

                self.holes[self.next_hole_id] = hole

                self.next_hole_id += 1
                self.holes_appeared += 1


    def fill_hole(self, position):

        for hole_id, hole in list(self.holes.items()):

            if hole.position == position:

                del self.holes[hole_id]

                self.holes_filled += 1

                return hole_id

        return None


# =========================================================
# BDI AGENT
# =========================================================

class BDIAgent:

    def __init__(
        self,
        world,
        gamma=1,
        verbose=True
    ):

        self.world = world

        self.position = (0, 0)

        # BDI components
        self.beliefs = {}
        self.desires = []
        self.intention = None
        self.plan_actions = []

        # Reconsideration interval
        self.gamma = gamma

        # Actions since last reconsideration
        self.actions_since_reconsideration = 0

        self.verbose = verbose


    # =====================================================
    # BRF - BELIEF REVISION FUNCTION
    # =====================================================

    def brf(self):

        self.beliefs = {
            hole_id: hole.position
            for hole_id, hole
            in self.world.holes.items()
        }


    # =====================================================
    # OPTIONS
    # =====================================================

    def options(self):

        self.desires = list(
            self.beliefs.keys()
        )

        return self.desires


    # =====================================================
    # FILTER
    # =====================================================

    def filter(self):

        if not self.desires:
            return None

        # Select nearest hole

        nearest_hole = min(
            self.desires,
            key=lambda hole_id:
            self.manhattan_distance(
                self.position,
                self.beliefs[hole_id]
            )
        )

        return nearest_hole


    # =====================================================
    # PLAN
    # =====================================================

    def plan(self):

        if self.intention is None:
            return []

        target = self.beliefs[
            self.intention
        ]

        x, y = self.position
        tx, ty = target

        actions = []

        while x < tx:
            actions.append("RIGHT")
            x += 1

        while x > tx:
            actions.append("LEFT")
            x -= 1

        while y < ty:
            actions.append("DOWN")
            y += 1

        while y > ty:
            actions.append("UP")
            y -= 1

        return actions


    # =====================================================
    # DISTANCE
    # =====================================================

    def manhattan_distance(self, a, b):

        return (
            abs(a[0] - b[0])
            +
            abs(a[1] - b[1])
        )


    # =====================================================
    # MOVE
    # =====================================================

    def execute_action(self, action):

        x, y = self.position

        if action == "RIGHT":
            x += 1

        elif action == "LEFT":
            x -= 1

        elif action == "DOWN":
            y += 1

        elif action == "UP":
            y -= 1

        self.position = (x, y)


    # =====================================================
    # COMMIT TO AN INTENTION
    # =====================================================

    def deliberate(self):

        self.brf()

        self.options()

        new_intention = self.filter()

        if new_intention is None:

            self.intention = None
            self.plan_actions = []

            return


        # Check whether intention changes

        if self.intention is None:

            self.intention = new_intention

            self.plan_actions = self.plan()

            if self.verbose:
                print(
                    f"COMMIT → Hole {self.intention} "
                    f"at {self.beliefs[self.intention]}"
                )

        elif new_intention != self.intention:

            old = self.intention

            self.intention = new_intention

            self.plan_actions = self.plan()

            if self.verbose:
                print(
                    f"SWITCH → Hole {old} "
                    f"to Hole {self.intention}"
                )


    # =====================================================
    # CHECK WHETHER INTENTION IS STILL ACHIEVABLE
    # =====================================================

    def reconsider(self):

        self.brf()

        # Current intention disappeared

        if (
            self.intention is not None
            and self.intention not in self.beliefs
        ):

            if self.verbose:
                print(
                    f"DROP → Hole {self.intention} "
                    f"is no longer available"
                )

            self.intention = None
            self.plan_actions = []

            return

        # Generate options again
        self.options()

        new_intention = self.filter()

        # Better hole exists

        if (
            new_intention is not None
            and new_intention != self.intention
        ):

            old = self.intention

            self.intention = new_intention

            self.plan_actions = self.plan()

            if self.verbose:
                print(
                    f"SWITCH → Hole {old} "
                    f"to Hole {new_intention}"
                )


    # =====================================================
    # ONE BDI STEP
    # =====================================================

    def step(self, world_speed):

        # -----------------------------------------------
        # Step 1: Perceive and revise beliefs
        # -----------------------------------------------

        self.brf()


        # -----------------------------------------------
        # Step 2: No intention or no plan
        # -----------------------------------------------

        if (
            self.intention is None
            or not self.plan_actions
        ):

            self.deliberate()


        # -----------------------------------------------
        # No hole available
        # -----------------------------------------------

        if self.intention is None:

            for _ in range(world_speed):
                self.world.tick()

            return


        # -----------------------------------------------
        # Execute one action
        # -----------------------------------------------

        # -----------------------------------------------
        # Execute next action
        # -----------------------------------------------

        if not self.plan_actions:

    # No action available.
    # Force fresh deliberation on the next cycle.

          self.intention = None
          self.plan_actions.clear()

          for _ in range(world_speed):
            self.world.tick()
          return

        action = self.plan_actions.pop(0)

        self.execute_action(action)

        self.actions_since_reconsideration += 1


        # -----------------------------------------------
        # World advances while agent acts
        # -----------------------------------------------

        for _ in range(world_speed):
            self.world.tick()


        # -----------------------------------------------
        # Update beliefs after world changes
        # -----------------------------------------------

        self.brf()


        # -----------------------------------------------
        # Check if hole was filled
        # -----------------------------------------------

        if self.intention in self.beliefs:

            target_position = self.beliefs[
                self.intention
            ]

            if self.position == target_position:

                filled = self.world.fill_hole(
                    self.position
                )

                if filled is not None:

                    if self.verbose:
                        print(
                            f"ACHIEVE → Hole {filled} filled"
                        )

                    self.intention = None
                    self.plan_actions = []


        # -----------------------------------------------
        # Reconsider after gamma actions
        # -----------------------------------------------

        if (
            self.actions_since_reconsideration
            >= self.gamma
        ):

            self.actions_since_reconsideration = 0

            # Deliberation takes time
            for _ in range(world_speed):
                self.world.tick()

            self.reconsider()


# =========================================================
# RUN SIMULATION
# =========================================================

def run_simulation(
    gamma=2,
    world_speed=1,
    steps=100,
    seed=42,
    verbose=True
):

    world = TileWorld(
        seed=seed
    )

    agent = BDIAgent(
        world,
        gamma=gamma,
        verbose=verbose
    )

    for step in range(steps):

        if verbose:
            print(
                f"\n--- Agent Step {step + 1} ---"
            )

        agent.step(world_speed)


    effectiveness = 0

    if world.holes_appeared > 0:

        effectiveness = (
            world.holes_filled
            /
            world.holes_appeared
        )


    print("\n========== RESULTS ==========")

    print("Gamma:", gamma)

    print(
        "Holes appeared:",
        world.holes_appeared
    )

    print(
        "Holes filled:",
        world.holes_filled
    )

    print(
        "Effectiveness:",
        round(effectiveness, 4)
    )

    return effectiveness

In [ ]:
run_simulation(
    gamma=2,
    world_speed=1,
    steps=100,
    seed=42,
    verbose=True
)


--- Agent Step 1 ---

--- Agent Step 2 ---

--- Agent Step 3 ---
COMMIT → Hole 1 at (4, 3)

--- Agent Step 4 ---

--- Agent Step 5 ---

--- Agent Step 6 ---

--- Agent Step 7 ---

--- Agent Step 8 ---

--- Agent Step 9 ---
ACHIEVE → Hole 1 filled

--- Agent Step 10 ---
COMMIT → Hole 2 at (1, 3)

--- Agent Step 11 ---

--- Agent Step 12 ---
ACHIEVE → Hole 2 filled
SWITCH → Hole None to Hole 3

--- Agent Step 13 ---

--- Agent Step 14 ---

--- Agent Step 15 ---

--- Agent Step 16 ---

--- Agent Step 17 ---

--- Agent Step 18 ---

--- Agent Step 19 ---

--- Agent Step 20 ---
DROP → Hole 3 is no longer available

--- Agent Step 21 ---
COMMIT → Hole 4 at (12, 2)

--- Agent Step 22 ---
DROP → Hole 4 is no longer available

--- Agent Step 23 ---

--- Agent Step 24 ---

--- Agent Step 25 ---
COMMIT → Hole 5 at (3, 12)

--- Agent Step 26 ---

--- Agent Step 27 ---

--- Agent Step 28 ---

--- Agent Step 29 ---
ACHIEVE → Hole 5 filled

--- Agent Step 30 ---

--- Agent Step 31 ---

--- Agent Step

0.5

In [3]:
def experiment_boldness():

    gammas = [1, 2, 4, 8, 10000]

    speeds = [1, 2, 4, 8]

    results = {}

    for speed in speeds:

        print(
            f"\n========== WORLD SPEED {speed} =========="
        )

        results[speed] = {}

        for gamma in gammas:

            effectiveness_values = []

            for seed in range(25):

                effectiveness = run_simulation(
                    gamma=gamma,
                    world_speed=speed,
                    steps=600,
                    seed=seed,
                    verbose=False
                )

                effectiveness_values.append(
                    effectiveness
                )


            average_effectiveness = (
                sum(effectiveness_values)
                /
                len(effectiveness_values)
            )

            results[speed][gamma] = (
                average_effectiveness
            )

            gamma_name = (
                "bold"
                if gamma == 10000
                else gamma
            )

            print(
                f"Gamma = {gamma_name}, "
                f"Average effectiveness = "
                f"{average_effectiveness:.4f}"
            )

    return results

In [ ]:
results = experiment_boldness()


========== WORLD SPEED 1 ==========

========== RESULTS ==========
Gamma: 1
Holes appeared: 75
Holes filled: 20
Effectiveness: 0.2667

========== RESULTS ==========
Gamma: 1
Holes appeared: 57
Holes filled: 12
Effectiveness: 0.2105

========== RESULTS ==========
Gamma: 1
Holes appeared: 75
Holes filled: 24
Effectiveness: 0.32

========== RESULTS ==========
Gamma: 1
Holes appeared: 72
Holes filled: 18
Effectiveness: 0.25

========== RESULTS ==========
Gamma: 1
Holes appeared: 78
Holes filled: 18
Effectiveness: 0.2308

========== RESULTS ==========
Gamma: 1
Holes appeared: 96
Holes filled: 28
Effectiveness: 0.2917

========== RESULTS ==========
Gamma: 1
Holes appeared: 70
Holes filled: 14
Effectiveness: 0.2

========== RESULTS ==========
Gamma: 1
Holes appeared: 86
Holes filled: 28
Effectiveness: 0.3256

========== RESULTS ==========
Gamma: 1
Holes appeared: 57
Holes filled: 14
Effectiveness: 0.2456

========== RESULTS ==========
Gamma: 1
Holes appeared: 98
Holes filled: 27
Effectivenes

In [ ]:
def print_results(results):

    gammas = [1, 2, 4, 8, 10000]

    print("\n")
    print("World Speed | γ=1 | γ=2 | γ=4 | γ=8 | Bold")

    print("-" * 65)

    for speed in results:

        values = []

        for gamma in gammas:

            values.append(
                f"{results[speed][gamma]:.4f}"
            )

        print(
            speed,
            " | ",
            " | ".join(values)
        )

print_results(results)



World Speed | γ=1 | γ=2 | γ=4 | γ=8 | Bold
-----------------------------------------------------------------
1  |  0.2639 | 0.3560 | 0.4068 | 0.4206 | 0.4179
2  |  0.0819 | 0.1291 | 0.1447 | 0.1376 | 0.1016
4  |  0.0239 | 0.0320 | 0.0337 | 0.0187 | 0.0172
8  |  0.0060 | 0.0052 | 0.0043 | 0.0035 | 0.0015


Slow environment
      ->
larger γ is better
      ->
World Speed 1 → γ = 8


Medium environment
      ->
intermediate γ is better
      ->
World Speed 2 → γ = 4
World Speed 4 → γ = 4


Fast environment
      ->
smaller γ is better
      ->
World Speed 8 → γ = 1



Conclusion

*   The experiment demonstrates the
importance of commitment and reconsideration in BDI agents.
*   In a slowly changing environment, larger γ values are preferable because the agent can safely remain committed to its current intention.




*  In a rapidly changing environment, smaller γ values are better because the agent needs to frequently reconsider outdated intentions.
*   The results support the Kinny–Georgeff commitment hypothesis:
Bold commitment → better in slowly changing worlds
Cautious reconsideration → better in rapidly changing worlds






*   Therefore, the reconsideration interval γ should be adapted according to the dynamism of the environment.




In [6]:
# =========================================================
# EXPERIMENT 2
# RESOURCE-BOUNDED BDI AGENT
# =========================================================

BATTERY_CAPACITY = 40
CHARGER_POSITION = (0, 0)


# =========================================================
# RESOURCE-BOUNDED BDI AGENT
# =========================================================

class ResourceBoundedBDIAgent:

    def __init__(self, world, verbose=True):

        self.world = world

        # Agent state
        self.position = CHARGER_POSITION
        self.battery = BATTERY_CAPACITY

        # BDI components
        self.beliefs = {}
        self.desires = []
        self.intention = None
        self.plan_actions = []

        # Statistics
        self.holes_filled = 0
        self.stranded_steps = 0
        self.total_moves = 0

        self.verbose = verbose

        # ---------------------------------------------
        # Plan library
        # ---------------------------------------------
        self.plan_library = {
            "fill-hole": self.plan_fill_hole,
            "recharge": self.plan_recharge
        }


    # =====================================================
    # BRF - BELIEF REVISION
    # =====================================================

    def brf(self):

        self.beliefs = {
            hole_id: hole.position
            for hole_id, hole in self.world.holes.items()
        }


    # =====================================================
    # OPTIONS
    # =====================================================

    def options(self):

        self.desires = list(self.beliefs.keys())

        return self.desires


    # =====================================================
    # DISTANCE
    # =====================================================

    def manhattan_distance(self, a, b):

        return (
            abs(a[0] - b[0])
            +
            abs(a[1] - b[1])
        )


    # =====================================================
    # PLAN SCHEMA 1: FILL-HOLE
    #
    # Applicable only when:
    #
    # battery >=
    # distance(current -> hole)
    # +
    # distance(hole -> charger)
    # =====================================================

    def fill_hole_applicable(self, hole_id):

        if hole_id not in self.beliefs:
            return False

        hole_position = self.beliefs[hole_id]

        distance_to_hole = self.manhattan_distance(
            self.position,
            hole_position
        )

        return_distance = self.manhattan_distance(
            hole_position,
            CHARGER_POSITION
        )

        required_battery = (
            distance_to_hole +
            return_distance
        )

        return self.battery >= required_battery


    # =====================================================
    # PLAN SCHEMA 2: RECHARGE
    #
    # Always applicable
    # =====================================================

    def recharge_applicable(self):

        return True


    # =====================================================
    # FILTER
    #
    # Select nearest FEASIBLE hole.
    # If none is feasible -> recharge.
    # =====================================================

    def filter(self):

        feasible_holes = []

        for hole_id in self.desires:

            if self.fill_hole_applicable(hole_id):

                distance = self.manhattan_distance(
                    self.position,
                    self.beliefs[hole_id]
                )

                feasible_holes.append(
                    (distance, hole_id)
                )


        # ---------------------------------------------
        # If at least one feasible hole exists,
        # select nearest feasible hole.
        # ---------------------------------------------

        if feasible_holes:

            feasible_holes.sort()

            nearest_distance, nearest_hole = (
                feasible_holes[0]
            )

            return ("fill-hole", nearest_hole)


        # ---------------------------------------------
        # No feasible hole -> recharge
        # ---------------------------------------------

        return ("recharge", None)


    # =====================================================
    # PLAN GENERATION
    # =====================================================

    def generate_path(self, target):

        x, y = self.position
        tx, ty = target

        actions = []

        # Horizontal movement
        while x < tx:
            actions.append("RIGHT")
            x += 1

        while x > tx:
            actions.append("LEFT")
            x -= 1

        # Vertical movement
        while y < ty:
            actions.append("DOWN")
            y += 1

        while y > ty:
            actions.append("UP")
            y -= 1

        return actions


    # =====================================================
    # PLAN SCHEMA: FILL-HOLE
    # =====================================================

    def plan_fill_hole(self, hole_id):

        if hole_id not in self.beliefs:
            return []

        target = self.beliefs[hole_id]

        return self.generate_path(target)


    # =====================================================
    # PLAN SCHEMA: RECHARGE
    # =====================================================

    def plan_recharge(self):

        return self.generate_path(
            CHARGER_POSITION
        )


    # =====================================================
    # PLAN
    # =====================================================

    def plan(self):

        if self.intention is None:
            return []

        schema, target = self.intention

        if schema == "fill-hole":

            return self.plan_fill_hole(target)

        elif schema == "recharge":

            return self.plan_recharge()

        return []


    # =====================================================
    # DELIBERATION
    # =====================================================

    def deliberate(self):

        self.brf()

        self.options()

        new_intention = self.filter()


        # ---------------------------------------------
        # Commit to intention
        # ---------------------------------------------

        self.intention = new_intention

        self.plan_actions = self.plan()


        if self.verbose:

            if new_intention[0] == "fill-hole":

                hole_id = new_intention[1]

                print(
                    f"COMMIT -> fill-hole "
                    f"Hole {hole_id} "
                    f"at {self.beliefs[hole_id]} "
                    f"| Battery = {self.battery}"
                )

            else:

                print(
                    f"COMMIT -> recharge "
                    f"| Battery = {self.battery}"
                )


    # =====================================================
    # EXECUTE ACTION
    # =====================================================

    def execute_action(self, action):

        # ---------------------------------------------
        # Cannot move without battery
        # ---------------------------------------------

        if self.battery <= 0:

            if self.position != CHARGER_POSITION:

                self.stranded_steps += 1

                return

            else:

                self.battery = BATTERY_CAPACITY


        x, y = self.position


        if action == "RIGHT":
            x += 1

        elif action == "LEFT":
            x -= 1

        elif action == "DOWN":
            y += 1

        elif action == "UP":
            y -= 1


        # ---------------------------------------------
        # Update position
        # ---------------------------------------------

        self.position = (x, y)

        # One battery consumed per move
        self.battery -= 1

        self.total_moves += 1


        # ---------------------------------------------
        # Recharge automatically at charger
        # ---------------------------------------------

        if self.position == CHARGER_POSITION:

            self.battery = BATTERY_CAPACITY

            if self.verbose:

                print(
                    f"RECHARGE -> Battery restored to "
                    f"{BATTERY_CAPACITY}"
                )


    # =====================================================
    # ACHIEVE INTENTION
    # =====================================================

    def check_achievement(self):

        if self.intention is None:
            return


        schema, target = self.intention


        # ---------------------------------------------
        # Fill-hole achievement
        # ---------------------------------------------

        if schema == "fill-hole":

            if target not in self.world.holes:

                self.intention = None
                self.plan_actions = []

                return


            hole_position = self.world.holes[
                target
            ].position


            if self.position == hole_position:

                filled = self.world.fill_hole(
                    self.position
                )

                if filled is not None:

                    self.holes_filled += 1

                    if self.verbose:

                        print(
                            f"ACHIEVE -> Hole {filled} filled "
                            f"| Battery = {self.battery}"
                        )

                    self.intention = None
                    self.plan_actions = []


        # ---------------------------------------------
        # Recharge achievement
        # ---------------------------------------------

        elif schema == "recharge":

            if self.position == CHARGER_POSITION:

                self.battery = BATTERY_CAPACITY

                if self.verbose:

                    print(
                        f"ACHIEVE -> Recharge complete "
                        f"| Battery = {self.battery}"
                    )

                self.intention = None
                self.plan_actions = []


    # =====================================================
    # ONE AGENT STEP
    # =====================================================

    def step(self, world_speed):

        # ---------------------------------------------
        # Update beliefs
        # ---------------------------------------------

        self.brf()


        # ---------------------------------------------
        # Deliberate if no intention/plan
        # ---------------------------------------------

        if (
            self.intention is None
            or not self.plan_actions
        ):

            self.deliberate()


        # ---------------------------------------------
        # If still no plan
        # ---------------------------------------------

        if not self.plan_actions:

            # If recharge intention and already at charger
            if (
                self.intention is not None
                and self.intention[0] == "recharge"
                and self.position == CHARGER_POSITION
            ):

                self.battery = BATTERY_CAPACITY

                self.intention = None
                self.plan_actions = []

            for _ in range(world_speed):
                self.world.tick()

            return


        # ---------------------------------------------
        # Execute one action
        # ---------------------------------------------

        action = self.plan_actions.pop(0)

        self.execute_action(action)


        # ---------------------------------------------
        # Check whether intention achieved
        # ---------------------------------------------

        self.check_achievement()


        # ---------------------------------------------
        # World advances
        # ---------------------------------------------

        for _ in range(world_speed):

            self.world.tick()


# =========================================================
# BATTERY-BLIND AGENT
# =========================================================

class BatteryBlindAgent:

    def __init__(self, world, verbose=True):

        self.world = world

        self.position = CHARGER_POSITION

        self.battery = BATTERY_CAPACITY

        self.holes_filled = 0
        self.stranded_steps = 0
        self.total_moves = 0

        self.plan_actions = []
        self.target_hole = None

        self.verbose = verbose


    # =====================================================
    # DISTANCE
    # =====================================================

    def manhattan_distance(self, a, b):

        return (
            abs(a[0] - b[0])
            +
            abs(a[1] - b[1])
        )


    # =====================================================
    # FIND NEAREST HOLE
    # =====================================================

    def nearest_hole(self):

        if not self.world.holes:

            return None

        return min(
            self.world.holes.values(),
            key=lambda hole:
            self.manhattan_distance(
                self.position,
                hole.position
            )
        )


    # =====================================================
    # CREATE PATH
    # =====================================================

    def generate_path(self, target):

        x, y = self.position
        tx, ty = target

        actions = []

        while x < tx:
            actions.append("RIGHT")
            x += 1

        while x > tx:
            actions.append("LEFT")
            x -= 1

        while y < ty:
            actions.append("DOWN")
            y += 1

        while y > ty:
            actions.append("UP")
            y -= 1

        return actions


    # =====================================================
    # EXECUTE ACTION
    # =====================================================

    def execute_action(self, action):

        # ---------------------------------------------
        # Battery-blind agent gets stranded
        # ---------------------------------------------

        if self.battery <= 0:

            if self.position != CHARGER_POSITION:

                self.stranded_steps += 1

                return

            else:

                self.battery = BATTERY_CAPACITY


        x, y = self.position


        if action == "RIGHT":
            x += 1

        elif action == "LEFT":
            x -= 1

        elif action == "DOWN":
            y += 1

        elif action == "UP":
            y -= 1


        self.position = (x, y)

        self.battery -= 1

        self.total_moves += 1


        # ---------------------------------------------
        # Charger restores battery
        # ---------------------------------------------

        if self.position == CHARGER_POSITION:

            self.battery = BATTERY_CAPACITY


    # =====================================================
    # ONE AGENT STEP
    # =====================================================

    def step(self, world_speed):

        # ---------------------------------------------
        # If no plan, choose nearest hole
        # ---------------------------------------------

        if not self.plan_actions:

            hole = self.nearest_hole()

            if hole is not None:

                self.target_hole = hole.id

                self.plan_actions = self.generate_path(
                    hole.position
                )


        # ---------------------------------------------
        # No hole available
        # ---------------------------------------------

        if not self.plan_actions:

            for _ in range(world_speed):
                self.world.tick()

            return


        # ---------------------------------------------
        # Execute movement
        # ---------------------------------------------

        action = self.plan_actions.pop(0)

        self.execute_action(action)


        # ---------------------------------------------
        # Check whether target hole is reached
        # ---------------------------------------------

        if self.target_hole in self.world.holes:

            hole = self.world.holes[
                self.target_hole
            ]

            if self.position == hole.position:

                filled = self.world.fill_hole(
                    self.position
                )

                if filled is not None:

                    self.holes_filled += 1

                    if self.verbose:

                        print(
                            f"BLIND ACHIEVE -> "
                            f"Hole {filled} filled "
                            f"| Battery = {self.battery}"
                        )

                self.target_hole = None
                self.plan_actions = []


        # ---------------------------------------------
        # If hole disappeared, abandon old plan
        # ---------------------------------------------

        elif self.target_hole not in self.world.holes:

            self.target_hole = None
            self.plan_actions = []


        # ---------------------------------------------
        # World advances
        # ---------------------------------------------

        for _ in range(world_speed):

            self.world.tick()


# =========================================================
# RUN RESOURCE-BOUNDED BDI
# =========================================================

def run_resource_bounded(
    steps=600,
    world_speed=1,
    seed=42,
    verbose=False
):

    world = TileWorld(seed=seed)

    agent = ResourceBoundedBDIAgent(
        world,
        verbose=verbose
    )


    for step in range(steps):

        agent.step(world_speed)


    return {
        "holes_appeared": world.holes_appeared,
        "holes_filled": agent.holes_filled,
        "stranded_steps": agent.stranded_steps,
        "total_moves": agent.total_moves
    }


# =========================================================
# RUN BATTERY-BLIND AGENT
# =========================================================

def run_battery_blind(
    steps=600,
    world_speed=1,
    seed=42,
    verbose=False
):

    world = TileWorld(seed=seed)

    agent = BatteryBlindAgent(
        world,
        verbose=verbose
    )


    for step in range(steps):

        agent.step(world_speed)


    return {
        "holes_appeared": world.holes_appeared,
        "holes_filled": agent.holes_filled,
        "stranded_steps": agent.stranded_steps,
        "total_moves": agent.total_moves
    }


# =========================================================
# SINGLE EXPERIMENT
# =========================================================

print("\n" + "=" * 60)
print("RESOURCE-BOUNDED BDI vs BATTERY-BLIND")
print("=" * 60)


SEED = 42
STEPS = 600
WORLD_SPEED = 1


resource_result = run_resource_bounded(
    steps=STEPS,
    world_speed=WORLD_SPEED,
    seed=SEED,
    verbose=False
)


blind_result = run_battery_blind(
    steps=STEPS,
    world_speed=WORLD_SPEED,
    seed=SEED,
    verbose=False
)


print("\nResource-Bounded BDI Agent")
print("--------------------------")
print(
    "Holes appeared:",
    resource_result["holes_appeared"]
)
print(
    "Holes filled:",
    resource_result["holes_filled"]
)
print(
    "Stranded steps:",
    resource_result["stranded_steps"]
)
print(
    "Total moves:",
    resource_result["total_moves"]
)


print("\nBattery-Blind Agent")
print("-------------------")
print(
    "Holes appeared:",
    blind_result["holes_appeared"]
)
print(
    "Holes filled:",
    blind_result["holes_filled"]
)
print(
    "Stranded steps:",
    blind_result["stranded_steps"]
)
print(
    "Total moves:",
    blind_result["total_moves"]
)


# =========================================================
# MULTI-SEED COMPARISON
# =========================================================

def compare_agents(
    num_seeds=25,
    steps=600,
    world_speed=1
):

    resource_filled = []
    resource_stranded = []

    blind_filled = []
    blind_stranded = []


    for seed in range(num_seeds):

        # ---------------------------------------------
        # Resource-bounded agent
        # ---------------------------------------------

        result1 = run_resource_bounded(
            steps=steps,
            world_speed=world_speed,
            seed=seed,
            verbose=False
        )

        resource_filled.append(
            result1["holes_filled"]
        )

        resource_stranded.append(
            result1["stranded_steps"]
        )


        # ---------------------------------------------
        # Battery-blind agent
        # ---------------------------------------------

        result2 = run_battery_blind(
            steps=steps,
            world_speed=world_speed,
            seed=seed,
            verbose=False
        )

        blind_filled.append(
            result2["holes_filled"]
        )

        blind_stranded.append(
            result2["stranded_steps"]
        )


    # ---------------------------------------------
    # Averages
    # ---------------------------------------------

    avg_resource_filled = (
        sum(resource_filled)
        / len(resource_filled)
    )

    avg_resource_stranded = (
        sum(resource_stranded)
        / len(resource_stranded)
    )

    avg_blind_filled = (
        sum(blind_filled)
        / len(blind_filled)
    )

    avg_blind_stranded = (
        sum(blind_stranded)
        / len(blind_stranded)
    )


    print("\n" + "=" * 60)
    print("AVERAGE RESULTS")
    print("=" * 60)

    print(
        f"\n{'Agent':<25}"
        f"{'Holes Filled':<20}"
        f"{'Stranded Steps':<20}"
    )

    print("-" * 65)

    print(
        f"{'Resource-Bounded BDI':<25}"
        f"{avg_resource_filled:<20.2f}"
        f"{avg_resource_stranded:<20.2f}"
    )

    print(
        f"{'Battery-Blind':<25}"
        f"{avg_blind_filled:<20.2f}"
        f"{avg_blind_stranded:<20.2f}"
    )


    return {
        "resource_filled": avg_resource_filled,
        "resource_stranded": avg_resource_stranded,
        "blind_filled": avg_blind_filled,
        "blind_stranded": avg_blind_stranded
    }


# =========================================================
# RUN 25-SEED EXPERIMENT
# =========================================================

comparison = compare_agents(
    num_seeds=25,
    steps=600,
    world_speed=1
)


RESOURCE-BOUNDED BDI vs BATTERY-BLIND

Resource-Bounded BDI Agent
--------------------------
Holes appeared: 48
Holes filled: 15
Stranded steps: 0
Total moves: 394

Battery-Blind Agent
-------------------
Holes appeared: 48
Holes filled: 3
Stranded steps: 353
Total moves: 40

AVERAGE RESULTS

Agent                    Holes Filled        Stranded Steps      
-----------------------------------------------------------------
Resource-Bounded BDI     16.16               0.00                
Battery-Blind            2.76                350.20              
